In [0]:
from pyspark.sql.functions import current_timestamp

CATALOG = "workspace"
SCHEMA = "retail"

def ingest_csv(file_name, table_name):
    path = f"/Volumes/{CATALOG}/default/{SCHEMA}/raw/{file_name}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")
    )

    print(f"{table_name} loaded successfully")
    

In [0]:
# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Generate sample data directly in volume using Spark
import random
from datetime import datetime, timedelta

# Create volume directory
volume_raw_path = f"/Volumes/{CATALOG}/default/retail/raw"
dbutils.fs.mkdirs(volume_raw_path)

# Generate customers data
customers_data = []
for cid in range(1, 1001):
    customers_data.append((
        cid,
        f"Customer_{cid}",
        f"customer{cid}@example.com",
        random.choice(["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]),
        random.choice(["NY", "CA", "IL", "TX", "AZ"]),
        (datetime.now() - timedelta(days=random.randint(0, 730))).strftime("%Y-%m-%d")
    ))

spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city", "state", "created_date"]) \
    .coalesce(1).write.mode("overwrite").option("header", True).csv(f"{volume_raw_path}/customers.csv")
print("customers.csv generated")

# Generate products data
products_data = []
categories = ["Electronics", "Fashion", "Home", "Sports"]
for pid in range(1, 201):
    products_data.append((
        pid,
        f"Product_{pid}",
        random.choice(categories),
        round(random.uniform(100, 5000), 2)
    ))

spark.createDataFrame(products_data, ["product_id", "product_name", "category", "price"]) \
    .coalesce(1).write.mode("overwrite").option("header", True).csv(f"{volume_raw_path}/products.csv")
print("products.csv generated")

# Generate orders data
orders_data = []
for oid in range(1, 5001):
    orders_data.append((
        oid,
        random.randint(1, 1000),
        random.randint(1, 200),
        random.randint(1, 5),
        (datetime.now() - timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d")
    ))

spark.createDataFrame(orders_data, ["order_id", "customer_id", "product_id", "quantity", "order_date"]) \
    .coalesce(1).write.mode("overwrite").option("header", True).csv(f"{volume_raw_path}/orders.csv")
print("orders.csv generated")

# Now ingest the data
ingest_csv("customers.csv", "bronze_customers")
ingest_csv("products.csv", "bronze_products")
ingest_csv("orders.csv", "bronze_orders")